# Transformer Models (BERT, GPT) for NLP
## AIAT 122 – Deep Learning

## Learning objectives
- Load BERT and get embeddings for a sentence (understanding).
- Load GPT-2 and generate a short text (generation).

## 🔗 Where this fits

**Builds on:** Course 07 (AIAT 121) — Unit 4, lesson 01 "Bridge: Attention and Transformers — Just Enough to Use Them" — it placed BERT and GPT on the transformer map; here you load both and see the difference in their outputs.

**Used later in:** Course 10 (AIAT 124) — Unit 2, lesson 02 "Fine-tuning Language Models".

---

**Where is this used in real life?** BERT powers search, classification, and NER; GPT powers chatbots and completion. **We use BERT for understanding** (bidirectional context) **and GPT for generation** (left-to-right); we use them instead of only RNNs because Transformers capture long-range dependencies in parallel and scale better.

**Prerequisites:** Basic PyTorch/transformers. Run `pip install transformers torch` if needed.

## Short theory
- **BERT:** Encoder-only; bidirectional; good for classification, NER, QA.
- **GPT:** Decoder-only; autoregressive; good for text generation.
- **Hugging Face:** Pre-trained models and tokenizers; we use them for embeddings and generation without training here.

**📌 Covers slide(s):** Optional — do after core notebooks 01–05 (no specific slide).


## Inputs & Outputs
**Inputs:** `transformers`, `torch`; pre-trained BERT and GPT-2 (downloaded on first run).  
**Dataset:** Real — pre-trained BERT and GPT-2 (Hugging Face; downloaded on first run).  
**Outputs:** BERT embedding shape for a sample sentence, and a short GPT-2 generated text. Run time: under ~5 min (download once).


In [1]:
# WHAT: install/import Hugging Face Transformers and load both BERT and GPT-2 classes.
# WHY: one library gives us an encoder (BERT) and a decoder (GPT-2) to compare side by side.
%pip install transformers torch -q
from transformers import BertTokenizer, BertModel, GPT2Tokenizer, GPT2LMHeadModel
import torch
print("✅ Imports OK.")

Note: you may need to restart the kernel to use updated packages.


✅ Imports OK.


### Step 1: BERT – get embeddings for a sentence

In [2]:
# We use BERT for understanding (bidirectional); good for classification/embeddings
bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
sentence = "Machine learning is a subset of artificial intelligence."
inputs = bert_tok(sentence, return_tensors="pt")
with torch.no_grad():
    out = bert(**inputs)
# [CLS] or mean of last_hidden_state = sentence embedding
emb = out.last_hidden_state
print("BERT output shape (batch, seq_len, hidden):", emb.shape)
print("Sentence embedding (mean over tokens) shape:", emb.mean(dim=1).shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT output shape (batch, seq_len, hidden): torch.Size([1, 11, 768])
Sentence embedding (mean over tokens) shape: torch.Size([1, 768])


### Step 2: GPT-2 – generate short text from a prompt

In [3]:
# We use GPT for generation (decoder-only, left-to-right)
gpt_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt_tok.pad_token = gpt_tok.eos_token
gpt = GPT2LMHeadModel.from_pretrained("gpt2")
prompt = "The future of AI is"
# Tokenize the prompt, generate 25 new tokens with sampling, then decode the IDs back to text.
inputs = gpt_tok(prompt, return_tensors="pt")
out = gpt.generate(inputs["input_ids"], max_new_tokens=25, do_sample=True, temperature=0.7, pad_token_id=gpt_tok.eos_token_id)
generated = gpt_tok.decode(out[0], skip_special_tokens=True)
print("Generated:", generated)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generated: The future of AI is not clear, but the most promising ones will be soon.

A lot of companies are starting to move toward AI,


## 🌍 Real-World Worked Example — Character-Level Text Generator

**Industry context:**
- GitHub Copilot launched in June 2021 "powered by OpenAI Codex" (GitHub blog, 29 June 2021). It has since moved to a choice of models, and GitHub does not publish which one answers any given request — so do not attach a model name to it from memory.
- Models in this family emit one **token** at a time, conditioned on everything before it. Not one character, and not a whole line at once.
- Autocomplete on your phone is a much smaller version of the same next-symbol loop.

We build a **character-level language model** that learns to generate text token by token — the exact mechanism behind all LLMs.

In [4]:
# WHAT: train a character-level LSTM language model on a Shakespeare passage, then sample new text.
# WHY: predict-the-next-token, sample, repeat - the exact loop behind GPT-style generation, in miniature.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
# ── Training text ────────────────────────────────────────────────────────────
text = (
    "to be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms against "
    "a sea of troubles and by opposing end them to die to sleep no more and by "
    "a sleep to say we end the heartache and the thousand natural shocks that "
    "flesh is heir to tis a consummation devoutly to be wished to die to sleep"
)

# Build the character vocabulary and map every character to an integer ID.
chars  = sorted(set(text))
c2i    = {c:i for i,c in enumerate(chars)}
i2c    = {i:c for c,i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]

# Training pairs: 20 characters of context -> the character that follows.
SEQ_LEN = 20
X_list, y_list = [], []
for i in range(len(enc)-SEQ_LEN-1):
    X_list.append(enc[i:i+SEQ_LEN])
    y_list.append(enc[i+SEQ_LEN])
X_t = torch.tensor(X_list, dtype=torch.long)
y_t = torch.tensor(y_list, dtype=torch.long)

# ── LSTM Language Model ───────────────────────────────────────────────────
# The model: embed characters, run a 2-layer LSTM, predict the next character from the last hidden state.
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out,_ = self.lstm(self.embed(x))
        return self.fc(out[:,-1,:])

model   = CharLM()
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

# 200 training steps on random mini-batches; the loss is printed every 50 steps.
for epoch in range(200):
    model.train()
    perm = torch.randperm(len(X_t))[:256]  # mini-batch
    loss = loss_fn(model(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch} — loss: {loss.item():.3f}")

# ── Text Generation (Greedy / Temperature Sampling) ──────────────────────
# Generation: feed the context, sample the next character from the softmax, append it, repeat.
def generate(seed_str, steps=80, temperature=0.8):
    model.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            # Temperature rescales the logits: lower = safer and more repetitive, higher = more random.
            logits = model(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = np.random.choice(len(probs), p=probs)
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

print("\n── Generated Text ──────────────────────────────────────────────")
print(generate("to be or not", steps=100))
print("\nThis is exactly how ChatGPT generates text — one token at a time.")

Epoch 0 — loss: 3.169


Epoch 50 — loss: 1.298


Epoch 100 — loss: 0.053


Epoch 150 — loss: 0.011



── Generated Text ──────────────────────────────────────────────
to be or notek a sea hir a slee to sa and by epos liog en end themr is to sleer sa nof sriu to thea tin the that

This is exactly how ChatGPT generates text — one token at a time.


## 🧩 Mini-exercise

**Try it:** Run BERT on a different sentence and print the embedding shape. Or generate a longer sequence with GPT-2 (e.g. max_length=50) and compare.

---

## Summary
**What you did:** Loaded BERT and got embedding shape for a sentence; loaded GPT-2 and generated a short continuation from a prompt.

**In real life you'd also:** Fine-tune BERT for classification, use GPT for longer generation with better decoding, and add task heads.

**The main idea:** BERT = understanding (bidirectional); GPT = generation (autoregressive); both are Transformer-based and pre-trained.

**Next:** `10_sentiment_analysis_translation_speech.ipynb` shows sentiment analysis with the pipeline API.

## 🔭 State of the field (2026)

Both halves of this notebook have current successors. On the encoder side, ModernBERT (arXiv 2412.13663) keeps BERT's objective but adds an 8,192-token context, 2 trillion training tokens and substantially lower memory use — a near drop-in replacement for `bert-base-uncased` in this workflow. On the decoder side the frontier stopped being dense: DeepSeek-V3 (arXiv 2412.19437) reaches frontier quality with a sparse mixture-of-experts network and multi-head latent attention, activating only a fraction of its parameters per token, with the training economics published.

**Why loading BERT and GPT-2 side by side is still the right exercise.** The choice this notebook forces — bidirectional encoder for understanding, causal decoder for generation — is the choice you actually make at work, and it is unchanged. A mixture-of-experts model is still a decoder; ModernBERT is still an encoder. Learn the two shapes and what each one's output actually is; the parameter counts will keep changing without you.


## 📚 References

1. Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS. https://arxiv.org/abs/1706.03762
2. Devlin, J., Chang, M.-W., Lee, K. & Toutanova, K. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL. https://arxiv.org/abs/1810.04805
3. Brown, T. B. et al. (2020). *Language Models are Few-Shot Learners* (GPT-3). NeurIPS. https://arxiv.org/abs/2005.14165
4. Hoffmann, J. et al. (2022). *Training Compute-Optimal Large Language Models* (Chinchilla). NeurIPS. https://arxiv.org/abs/2203.15556

**Recent work (2024–2026)**

5. DeepSeek-AI (2024). *DeepSeek-V3 Technical Report*. arXiv. https://arxiv.org/abs/2412.19437 — where the GPT side went: a sparse mixture-of-experts model with multi-head latent attention at frontier quality, with its training economics published.
6. Warner, B. et al. (2024). *Smarter, Better, Faster, Longer: A Modern Bidirectional Encoder for Fast, Memory Efficient, and Long Context Finetuning and Inference* (ModernBERT). arXiv. https://arxiv.org/abs/2412.13663 — and where the BERT side went: the same encoder objective, 8,192-token context, 2 trillion training tokens, much lower memory.
